In [0]:
# CELDA 0: Inicialización para Databricks (nueva versión con Volumes)
import os

EN_DATABRICKS = True  # Estamos en Databricks

# Ruta del CSV en Volumes
CSV_PATH = "/Volumes/workspace/default/datasets/netflix_titles.csv"

print("✅ Entorno: Databricks — Serverless")
print(f"   Spark Version : {spark.version}")
print(f"   CSV Path      : {CSV_PATH}")

# Verificar que el CSV existe
try:
    files = dbutils.fs.ls("/Volumes/workspace/default/datasets/")
    for f in files:
        print(f"   📄 {f.name}  |  {f.size/1024:.1f} KB")
    print("✅ CSV encontrado correctamente")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Entorno: Databricks — Serverless
   Spark Version : 4.1.0
   CSV Path      : /Volumes/workspace/default/datasets/netflix_titles.csv
   📄 netflix_titles.csv  |  3320.0 KB
✅ CSV encontrado correctamente


# 🏗️ Taller: Procesamiento de Datos en una Infraestructura Cloud
## Databricks Community Edition — Actividad 2

---

| Campo | Detalle |
|---|---|
| **Asignatura** | Infraestructura y Bases de Datos en la Nube |
| **Actividad** | Actividad 2 — Procesamiento de datos en infraestructura cloud |
| **Plataforma** | Databricks Community Edition |
| **Dataset** | Netflix Movies and TV Shows (Kaggle) |
| **Herramientas** | PySpark, Spark SQL, DBFS |

---

## 📋 Objetivos

1. **Diseñar** el esquema que almacenará los datos (StructType / DDL).
2. **Configurar y evidenciar** el entorno en Databricks CE.
3. **Ingerir datos** desde Kaggle y crear una tabla persistente.
4. **Validar** con consultas de metadatos, descripción, SELECT y GROUP BY en Spark y SQL.
5. **Analizar** ventajas y desventajas de SQL vs Spark.

---

## 🗺️ Índice

1. [Diseño del Esquema](#1-diseño-del-esquema)
2. [Configuración de la Infraestructura](#2-configuración-de-la-infraestructura)
3. [Obtención e Ingesta de Datos](#3-obtención-e-ingesta-de-datos)
4. [Validaciones en Spark y SQL](#4-validaciones-en-spark-y-sql)
5. [Análisis: SQL vs Spark](#5-análisis-sql-vs-spark)
6. [Conclusiones](#6-conclusiones)

---
## ⚙️ Celda 0 — Inicialización Local (solo VSCode)

> **Nota:** Esta celda inicializa Spark en modo local para pruebas en VSCode. En Databricks, `spark` ya existe automáticamente y esta celda se omite.

In [0]:
# CELDA 0: Inicialización LOCAL para VSCode
# Esta celda NO va en Databricks — solo para pruebas locales
import os
from pyspark.sql import SparkSession

# Detectar entorno
try:
    dbutils
    EN_DATABRICKS = True
    print("✅ Entorno: Databricks CE — spark ya está disponible")
except NameError:
    EN_DATABRICKS = False
    spark = SparkSession.builder \
        .appName("Netflix-Actividad2-Local") \
        .master("local[*]") \
        .config("spark.sql.shuffle.partitions", "4") \
        .config("spark.driver.memory", "4g") \
        .getOrCreate()
    spark.sparkContext.setLogLevel("ERROR")
    print("✅ Entorno: Local / VSCode — SparkSession creada")

CSV_PATH = "./data/netflix_titles.csv"
os.makedirs("./data", exist_ok=True)

print(f"   Spark Version : {spark.version}")
print(f"   Master        : {spark.sparkContext.master}")
print(f"   CSV existe    : {os.path.exists(CSV_PATH)}")

✅ Entorno: Local / VSCode — SparkSession creada
   Spark Version : 3.4.1
   Master        : local[*]
   CSV existe    : True


---
# 1. Diseño del Esquema

## 1.1 Dataset Seleccionado

**Dataset:** [Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows)  
**Fuente:** Kaggle — usuario *shivamb*  
**Formato:** CSV | **Registros:** ~8,800 filas

---

## 1.2 Diccionario de Datos

| # | Campo | Tipo Spark | Nulable | Descripción |
|---|---|---|---|---|
| 1 | `show_id` | StringType | NO | Identificador único (PK) |
| 2 | `type` | StringType | NO | Movie o TV Show |
| 3 | `title` | StringType | NO | Nombre del título |
| 4 | `director` | StringType | SÍ | Director(es) |
| 5 | `cast` | StringType | SÍ | Elenco principal |
| 6 | `country` | StringType | SÍ | País de origen |
| 7 | `date_added` | StringType | SÍ | Fecha de ingreso a Netflix |
| 8 | `release_year` | IntegerType | NO | Año de estreno |
| 9 | `rating` | StringType | SÍ | Clasificación (PG-13, TV-MA…) |
| 10 | `duration` | StringType | SÍ | Duración (90 min / 3 Seasons) |
| 11 | `listed_in` | StringType | SÍ | Géneros/categorías |
| 12 | `description` | StringType | SÍ | Sinopsis |

---

## 1.3 Diagrama ER (Mermaid)

```mermaid
erDiagram
    NETFLIX_CONTENT {
        STRING show_id PK
        STRING type
        STRING title
        STRING director
        STRING cast
        STRING country
        STRING date_added
        INT    release_year
        STRING rating
        STRING duration
        STRING listed_in
        STRING description
    }
```

---

## 1.4 DDL Spark SQL

```sql
CREATE TABLE IF NOT EXISTS netflix_content (
    show_id      STRING  NOT NULL COMMENT 'ID único del título',
    type         STRING  NOT NULL COMMENT 'Movie o TV Show',
    title        STRING  NOT NULL COMMENT 'Nombre del título',
    director     STRING           COMMENT 'Director(es)',
    cast         STRING           COMMENT 'Actores principales',
    country      STRING           COMMENT 'País de producción',
    date_added   STRING           COMMENT 'Fecha de incorporación a Netflix',
    release_year INT     NOT NULL COMMENT 'Año de estreno',
    rating       STRING           COMMENT 'Clasificación por edad',
    duration     STRING           COMMENT 'Duración en min o temporadas',
    listed_in    STRING           COMMENT 'Géneros y categorías',
    description  STRING           COMMENT 'Sinopsis'
) USING delta
COMMENT 'Catálogo de contenido de Netflix — Dataset Kaggle';
```

In [0]:
# CELDA 1: Definición del esquema StructType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

netflix_schema = StructType([
    StructField("show_id",      StringType(),  nullable=False),
    StructField("type",         StringType(),  nullable=False),
    StructField("title",        StringType(),  nullable=False),
    StructField("director",     StringType(),  nullable=True),
    StructField("cast",         StringType(),  nullable=True),
    StructField("country",      StringType(),  nullable=True),
    StructField("date_added",   StringType(),  nullable=True),
    StructField("release_year", IntegerType(), nullable=False),
    StructField("rating",       StringType(),  nullable=True),
    StructField("duration",     StringType(),  nullable=True),
    StructField("listed_in",    StringType(),  nullable=True),
    StructField("description",  StringType(),  nullable=True),
])

print("✅ Esquema StructType definido:")
print("=" * 52)
for f in netflix_schema.fields:
    n = "NOT NULL" if not f.nullable else "NULLABLE"
    print(f"  {f.name:<16} {str(f.dataType):<15} [{n}]")
print("=" * 52)
print(f"Total campos: {len(netflix_schema.fields)}")

✅ Esquema StructType definido:
  show_id          StringType()    [NOT NULL]
  type             StringType()    [NOT NULL]
  title            StringType()    [NOT NULL]
  director         StringType()    [NULLABLE]
  cast             StringType()    [NULLABLE]
  country          StringType()    [NULLABLE]
  date_added       StringType()    [NULLABLE]
  release_year     IntegerType()   [NOT NULL]
  rating           StringType()    [NULLABLE]
  duration         StringType()    [NULLABLE]
  listed_in        StringType()    [NULLABLE]
  description      StringType()    [NULLABLE]
Total campos: 12


---
# 2. Configuración de la Infraestructura en Databricks CE

## 2.1 Pasos de Configuración del Clúster

| Paso | Acción | Valor |
|---|---|---|
| 1 | Ir a | `Compute → Create Cluster` |
| 2 | Cluster Name | `cluster-actividad2` |
| 3 | Cluster Mode | `Single Node` (único en CE) |
| 4 | Databricks Runtime | `13.3 LTS (Spark 3.4.1, Scala 2.12)` |
| 5 | Node Type | `15.3 GB Memory, 2 Cores` |
| 6 | Autoscaling | ❌ No disponible en CE |

## 2.2 Resumen de Configuración

| Parámetro | Valor |
|---|---|
| **Runtime** | 13.3 LTS |
| **Apache Spark** | 3.4.1 |
| **Scala** | 2.12 |
| **Python** | 3.10.x |
| **RAM** | 15.3 GB |
| **Núcleos** | 2 vCPUs |
| **Almacenamiento** | DBFS `/FileStore/` |

In [0]:
# CELDA 2: Versiones del entorno — Databricks Serverless
import sys

print("=" * 52)
print("  CONFIGURACIÓN DEL ENTORNO — DATABRICKS SERVERLESS")
print("=" * 52)
print(f"🐍 Python         : {sys.version.split()[0]}")
print(f"⚡ Spark          : {spark.version}")
print(f"☁️  Modo           : Databricks Serverless")
print(f"📦 Catálogo       : workspace")
print(f"🗄️  Schema         : default")
print(f"📁 Volume         : /Volumes/workspace/default/datasets/")
print("=" * 52)

  CONFIGURACIÓN DEL ENTORNO — DATABRICKS SERVERLESS
🐍 Python         : 3.11.10
⚡ Spark          : 4.1.0
☁️  Modo           : Databricks Serverless
📦 Catálogo       : workspace
🗄️  Schema         : default
📁 Volume         : /Volumes/workspace/default/datasets/


In [0]:
# CELDA 2B: Evidencia del entorno Serverless
print("╔" + "═"*50 + "╗")
print("║   EVIDENCIA DE INFRAESTRUCTURA — SERVERLESS      ║")
print("╚" + "═"*50 + "╝")
print(f"  Spark Version     : {spark.version}")
print(f"  Python Version    : {sys.version.split()[0]}")
print(f"  Modo              : Databricks Serverless")
print(f"  Almacenamiento    : Unity Catalog Volumes")
print(f"  CSV Path          : /Volumes/workspace/default/datasets/netflix_titles.csv")
print("-" * 52)
print("⚙️  Configuraciones Spark disponibles:")
claves = [
    "spark.sql.shuffle.partitions",
    "spark.sql.ansi.enabled",
    "spark.sql.adaptive.enabled"
]
for k in claves:
    try:
        v = spark.conf.get(k)
        print(f"  {k.split('.')[-1]:<30} = {v}")
    except:
        print(f"  {k.split('.')[-1]:<30} = (no disponible en Serverless)")
print("╔" + "═"*50 + "╗")
print("║  ✅ Evidencia registrada correctamente            ║")
print("╚" + "═"*50 + "╝")

╔══════════════════════════════════════════════════╗
║   EVIDENCIA DE INFRAESTRUCTURA — SERVERLESS      ║
╚══════════════════════════════════════════════════╝
  Spark Version     : 4.1.0
  Python Version    : 3.11.10
  Modo              : Databricks Serverless
  Almacenamiento    : Unity Catalog Volumes
  CSV Path          : /Volumes/workspace/default/datasets/netflix_titles.csv
----------------------------------------------------
⚙️  Configuraciones Spark disponibles:
  partitions                     = auto
  enabled                        = true
  enabled                        = (no disponible en Serverless)
╔══════════════════════════════════════════════════╗
║  ✅ Evidencia registrada correctamente            ║
╚══════════════════════════════════════════════════╝


In [0]:
# CELDA 3: Estructura de almacenamiento — Unity Catalog Volumes
print("📁 ESTRUCTURA DE ALMACENAMIENTO — UNITY CATALOG")
print("=" * 52)
print("  Entorno    : Databricks Serverless")
print("  Sistema    : Unity Catalog (reemplaza DBFS en nueva versión)")
print("  Ruta       : /Volumes/workspace/default/datasets/")
print()

# Listar archivos en el Volume
print("📂 Contenido del Volume 'datasets':")
try:
    files = dbutils.fs.ls("/Volumes/workspace/default/datasets/")
    for f in files:
        size_mb = f.size / (1024 * 1024)
        print(f"  📄 {f.name}  |  {size_mb:.2f} MB  |  {f.path}")
except Exception as e:
    print(f"  Error: {e}")

print()
print("📊 Estructura Unity Catalog:")
print("  workspace")
print("  └── default")
print("      └── datasets  (Volume)")
print("          └── netflix_titles.csv")
print("=" * 52)
print("🎯 Ruta de trabajo: /Volumes/workspace/default/datasets/")

📁 ESTRUCTURA DE ALMACENAMIENTO — UNITY CATALOG
  Entorno    : Databricks Serverless
  Sistema    : Unity Catalog (reemplaza DBFS en nueva versión)
  Ruta       : /Volumes/workspace/default/datasets/

📂 Contenido del Volume 'datasets':
  📄 netflix_titles.csv  |  3.24 MB  |  dbfs:/Volumes/workspace/default/datasets/netflix_titles.csv

📊 Estructura Unity Catalog:
  workspace
  └── default
      └── datasets  (Volume)
          └── netflix_titles.csv
🎯 Ruta de trabajo: /Volumes/workspace/default/datasets/


---
# 3. Obtención e Ingesta de Datos desde Kaggle

## 3.1 Estrategia de Descarga

Utilizamos la **API de Kaggle** para automatizar la descarga del dataset.

**Dataset:** [Netflix Movies and TV Shows](https://www.kaggle.com/datasets/shivamb/netflix-shows)

### Flujo de Ingesta:
```
Kaggle API → DBFS /FileStore/ → spark.read.csv() → DataFrame → saveAsTable()
```

### Prerrequisitos Kaggle API:
1. Cuenta en [kaggle.com](https://www.kaggle.com)
2. `Account → Create New Token → kaggle.json`

In [0]:
# CELDA 4: Instalación de Kaggle (solo Databricks)
# En local ya lo instalaste con pip install kaggle
try:
    dbutils
    import subprocess
    subprocess.run(["pip", "install", "kaggle", "--quiet"], check=True)
    print("✅ Kaggle instalado en clúster Databricks")
except NameError:
    import kaggle
    print("✅ Kaggle ya disponible en entorno local")

✅ Kaggle instalado en clúster Databricks



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [0]:
# CELDA 5: Configuración del token de Kaggle
import os, json

# ⚠️ REEMPLAZA con tus credenciales reales de Kaggle
KAGGLE_USERNAME = "tu_usuario_kaggle"   # <-- Cambiar
KAGGLE_KEY      = "tu_api_key_kaggle"   # <-- Cambiar

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)
kaggle_json = os.path.join(kaggle_dir, "kaggle.json")
with open(kaggle_json, "w") as f:
    json.dump({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}, f)
os.chmod(kaggle_json, 0o600)

print(f"✅ Token configurado: {kaggle_json}")
print(f"   Usuario : {KAGGLE_USERNAME}")

✅ Token configurado: /home/spark-dbeecaea-a1c9-4e78-bbaf-03/.kaggle/kaggle.json
   Usuario : tu_usuario_kaggle


In [0]:
# CELDA 6: Descarga del dataset desde Kaggle
import subprocess, os

LOCAL_DIR = "/tmp/netflix_data/" if os.path.exists("/tmp") else "./data/"
os.makedirs(LOCAL_DIR, exist_ok=True)

print("⬇️  Descargando dataset desde Kaggle...")
result = subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "shivamb/netflix-shows", "--unzip", "-p", LOCAL_DIR],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Descarga exitosa")
else:
    print(f"❌ Error: {result.stderr}")

print("\n📂 Archivos descargados:")
for f in os.listdir(LOCAL_DIR):
    size = os.path.getsize(os.path.join(LOCAL_DIR, f))
    print(f"  📄 {f}  ({size/1024:.1f} KB)")

⬇️  Descargando dataset desde Kaggle...
✅ Descarga exitosa

📂 Archivos descargados:
  📄 netflix_titles.csv  (3320.0 KB)


In [0]:
# CELDA 7: Verificación del CSV en Unity Catalog Volume
# (El CSV fue subido manualmente al Volume — no se requiere copia)
print("✅ CSV ya disponible en Unity Catalog Volume")
print(f"   Ruta: /Volumes/workspace/default/datasets/netflix_titles.csv")

files = dbutils.fs.ls("/Volumes/workspace/default/datasets/")
for f in files:
    print(f"  📄 {f.name}  |  {f.size/1024/1024:.2f} MB")

✅ CSV ya disponible en Unity Catalog Volume
   Ruta: /Volumes/workspace/default/datasets/netflix_titles.csv
  📄 netflix_titles.csv  |  3.24 MB


In [0]:
# CELDA 8: Lectura del CSV con Spark y esquema explícito
import os

# Ruta según entorno
try:
    dbutils
    DBFS_CSV_PATH = "/Volumes/workspace/default/datasets/netflix_titles.csv"
except NameError:
    DBFS_CSV_PATH = "./data/netflix_titles.csv"

print(f"📂 Leyendo desde: {DBFS_CSV_PATH}")

df_netflix = (
    spark.read
    .format("csv")
    .option("header",    "true")
    .option("multiLine", "true")
    .option("quote",     '"')
    .option("escape",    '"')
    .option("encoding",  "UTF-8")
    .option("mode",      "PERMISSIVE")
    .schema(netflix_schema)
    .load(DBFS_CSV_PATH)
)

total_rows    = df_netflix.count()
total_columns = len(df_netflix.columns)

print("✅ DataFrame cargado exitosamente")
print("=" * 52)
print(f"  📊 Registros  : {total_rows:,}")
print(f"  📋 Columnas   : {total_columns}")
print("=" * 52)
df_netflix.printSchema()

📂 Leyendo desde: /Volumes/workspace/default/datasets/netflix_titles.csv
✅ DataFrame cargado exitosamente
  📊 Registros  : 8,807
  📋 Columnas   : 12
root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)



In [0]:
# CELDA 9: Vista previa de los datos
print("📋 Primeros 5 registros:")
df_netflix.show(5, truncate=45)

print("\n📋 Primer registro (detallado):")
df_netflix.show(1, truncate=False, vertical=True)

📋 Primeros 5 registros:
+-------+-------+---------------------+---------------+---------------------------------------------+-------------+------------------+------------+------+---------+---------------------------------------------+---------------------------------------------+
|show_id|   type|                title|       director|                                         cast|      country|        date_added|release_year|rating| duration|                                    listed_in|                                  description|
+-------+-------+---------------------+---------------+---------------------------------------------+-------------+------------------+------------+------+---------+---------------------------------------------+---------------------------------------------+
|     s1|  Movie| Dick Johnson Is Dead|Kirsten Johnson|                                         NULL|United States|September 25, 2021|        2020| PG-13|   90 min|                                Documenta

In [0]:
# CELDA 10: Persistencia de la tabla
TABLE_NAME = "netflix_content"

try:
    dbutils
    EN_DATABRICKS = True
except NameError:
    EN_DATABRICKS = False

if EN_DATABRICKS:
    spark.sql(f"DROP TABLE IF EXISTS {TABLE_NAME}")
    (
        df_netflix.write
        .mode("overwrite")
        .format("delta")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE_NAME)
    )
    print(f"✅ Tabla '{TABLE_NAME}' creada en Hive Metastore (Delta Lake)")
else:
    df_netflix.createOrReplaceTempView(TABLE_NAME)
    print(f"✅ Vista temporal '{TABLE_NAME}' creada (modo local)")

print(f"   Registros : {total_rows:,}")
print(f"   Entorno   : {'Databricks CE' if EN_DATABRICKS else 'Local / VSCode'}")
print("\n📋 Tablas disponibles:")
spark.sql("SHOW TABLES").show()

✅ Tabla 'netflix_content' creada en Hive Metastore (Delta Lake)
   Registros : 8,807
   Entorno   : Databricks CE

📋 Tablas disponibles:
+--------+---------------+-----------+
|database|      tableName|isTemporary|
+--------+---------------+-----------+
| default|netflix_content|      false|
+--------+---------------+-----------+



In [0]:
# CELDA 10B: Confirmación post-creación (PySpark)
df_from_table = spark.table(TABLE_NAME)

print("╔" + "═"*50 + "╗")
print("║   CONFIRMACIÓN DE INGESTA — TABLA CREADA         ║")
print("╚" + "═"*50 + "╝")
print(f"\n✅ Tabla leída desde: '{TABLE_NAME}'")
print(f"   Registros : {df_from_table.count():,}")
print(f"   Columnas  : {len(df_from_table.columns)}")
print("\n📋 Esquema persistido:")
df_from_table.printSchema()
print("\n📋 Muestra (3 filas):")
df_from_table.select("show_id","type","title","release_year","country").show(3, truncate=35)

╔══════════════════════════════════════════════════╗
║   CONFIRMACIÓN DE INGESTA — TABLA CREADA         ║
╚══════════════════════════════════════════════════╝

✅ Tabla leída desde: 'netflix_content'
   Registros : 8,807
   Columnas  : 12

📋 Esquema persistido:
root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)


📋 Muestra (3 filas):
+-------+-------+--------------------+------------+-------------+
|show_id|   type|               title|release_year|      country|
+-------+-------+--------------------+------------+-------------+
|     s1|  Movie|Dick Johnson Is Dead

In [0]:
# CELDA 10C: DESCRIBE TABLE (Spark SQL)
spark.sql(f"DESCRIBE TABLE {TABLE_NAME}").show(truncate=False)

+------------+---------+-------+
|col_name    |data_type|comment|
+------------+---------+-------+
|show_id     |string   |NULL   |
|type        |string   |NULL   |
|title       |string   |NULL   |
|director    |string   |NULL   |
|cast        |string   |NULL   |
|country     |string   |NULL   |
|date_added  |string   |NULL   |
|release_year|int      |NULL   |
|rating      |string   |NULL   |
|duration    |string   |NULL   |
|listed_in   |string   |NULL   |
|description |string   |NULL   |
+------------+---------+-------+



---
# 4. Validaciones en Spark y SQL

## 4.1 Objetivo

Garantizar la calidad e integridad de los datos comparando PySpark y Spark SQL.

In [0]:
# CELDA 11: Validación 1A — Metadatos PySpark
print("🔍 VALIDACIÓN 1A: printSchema() — Metadatos del DataFrame")
print("-" * 52)
df_netflix.printSchema()

print("\n📊 Tipos de datos por columna:")
print(f"{'Columna':<20} {'Tipo'}")
print("-" * 35)
for name, dtype in df_netflix.dtypes:
    print(f"  {name:<20} {dtype}")

🔍 VALIDACIÓN 1A: printSchema() — Metadatos del DataFrame
----------------------------------------------------
root
 |-- show_id: string (nullable = true)
 |-- type: string (nullable = true)
 |-- title: string (nullable = true)
 |-- director: string (nullable = true)
 |-- cast: string (nullable = true)
 |-- country: string (nullable = true)
 |-- date_added: string (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- rating: string (nullable = true)
 |-- duration: string (nullable = true)
 |-- listed_in: string (nullable = true)
 |-- description: string (nullable = true)


📊 Tipos de datos por columna:
Columna              Tipo
-----------------------------------
  show_id              string
  type                 string
  title                string
  director             string
  cast                 string
  country              string
  date_added           string
  release_year         int
  rating               string
  duration             string
  listed_in       

In [0]:
# CELDA 12: Validación 1B — Metadatos SQL
print("🔍 VALIDACIÓN 1B: DESCRIBE TABLE (Spark SQL)")
print("-" * 52)
spark.sql(f"DESCRIBE TABLE {TABLE_NAME}").show(truncate=False)

🔍 VALIDACIÓN 1B: DESCRIBE TABLE (Spark SQL)
----------------------------------------------------
+------------+---------+-------+
|col_name    |data_type|comment|
+------------+---------+-------+
|show_id     |string   |NULL   |
|type        |string   |NULL   |
|title       |string   |NULL   |
|director    |string   |NULL   |
|cast        |string   |NULL   |
|country     |string   |NULL   |
|date_added  |string   |NULL   |
|release_year|int      |NULL   |
|rating      |string   |NULL   |
|duration    |string   |NULL   |
|listed_in   |string   |NULL   |
|description |string   |NULL   |
+------------+---------+-------+



In [0]:
# CELDA 13: Validación 2A — Estadísticas descriptivas PySpark
print("🔍 VALIDACIÓN 2A: df.describe() — Estadísticas básicas")
print("-" * 52)
df_netflix.describe("show_id","type","release_year","rating","duration").show(truncate=30)

🔍 VALIDACIÓN 2A: df.describe() — Estadísticas básicas
----------------------------------------------------
+-------+-------+-------+------------------+------+--------+
|summary|show_id|   type|      release_year|rating|duration|
+-------+-------+-------+------------------+------+--------+
|  count|   8807|   8807|              8807|  8803|    8804|
|   mean|   NULL|   NULL|2014.1801975701146|  NULL|    NULL|
| stddev|   NULL|   NULL| 8.819312130833964|  NULL|    NULL|
|    min|     s1|  Movie|              1925|66 min|1 Season|
|    max|   s999|TV Show|              2021|    UR|  99 min|
+-------+-------+-------+------------------+------+--------+



In [0]:
# CELDA 14: Validación 2B — Análisis de nulos por columna
from pyspark.sql import functions as F

print("🔍 VALIDACIÓN 2B: Valores nulos por columna")
print("-" * 52)

null_expr = [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_netflix.columns]
null_counts = df_netflix.select(null_expr).collect()[0].asDict()

print(f"{'Columna':<20} {'Nulos':>8} {'% Nulos':>10}")
print("-" * 42)
for col, n in sorted(null_counts.items(), key=lambda x: x[1], reverse=True):
    pct = (n / total_rows) * 100
    bar = "█" * int(pct / 5)
    print(f"  {col:<20} {n:>8,} {pct:>9.1f}%  {bar}")

🔍 VALIDACIÓN 2B: Valores nulos por columna
----------------------------------------------------
Columna                 Nulos    % Nulos
------------------------------------------
  director                2,634      29.9%  █████
  country                   831       9.4%  █
  cast                      825       9.4%  █
  date_added                 10       0.1%  
  rating                      4       0.0%  
  duration                    3       0.0%  
  show_id                     0       0.0%  
  type                        0       0.0%  
  title                       0       0.0%  
  release_year                0       0.0%  
  listed_in                   0       0.0%  
  description                 0       0.0%  


In [0]:
# CELDA 15: Validación 2C — Estadísticas SQL
print("🔍 VALIDACIÓN 2C: Estadísticas con Spark SQL")
spark.sql(f"""
SELECT
    COUNT(*)                    AS total_registros,
    COUNT(DISTINCT show_id)     AS ids_unicos,
    MIN(release_year)           AS anio_min,
    MAX(release_year)           AS anio_max,
    ROUND(AVG(release_year),2)  AS anio_promedio,
    COUNT(CASE WHEN director IS NULL THEN 1 END) AS directores_nulos,
    COUNT(CASE WHEN country  IS NULL THEN 1 END) AS paises_nulos
FROM {TABLE_NAME}
""").show()

🔍 VALIDACIÓN 2C: Estadísticas con Spark SQL
+---------------+----------+--------+--------+-------------+----------------+------------+
|total_registros|ids_unicos|anio_min|anio_max|anio_promedio|directores_nulos|paises_nulos|
+---------------+----------+--------+--------+-------------+----------------+------------+
|           8807|      8807|    1925|    2021|      2014.18|            2634|         831|
+---------------+----------+--------+--------+-------------+----------------+------------+



In [0]:
# CELDA 16: Validación 3A — SELECT con filtros PySpark
print("🔍 VALIDACIÓN 3A: SELECT con filtros (PySpark)")
print("-" * 52)
result = (
    df_netflix
    .filter(
        (F.col("type") == "Movie") &
        (F.col("release_year") >= 2015) &
        (F.col("country").like("%United States%"))
    )
    .select("title","release_year","rating","duration")
    .orderBy(F.col("release_year").desc())
)
print(f"  Registros encontrados: {result.count():,}")
result.show(8, truncate=40)

🔍 VALIDACIÓN 3A: SELECT con filtros (PySpark)
----------------------------------------------------
  Registros encontrados: 1,662
+------------------------------------+------------+------+--------+
|                               title|release_year|rating|duration|
+------------------------------------+------------+------+--------+
|Monster Hunter: Legends of the Guild|        2021| TV-PG|  59 min|
|              Untold: Breaking Point|        2021| TV-MA|  80 min|
|                        The Starling|        2021| PG-13| 104 min|
|                 The Kissing Booth 3|        2021| TV-14| 114 min|
|                          Sweet Girl|        2021|     R| 110 min|
|                       Final Account|        2021| PG-13|  94 min|
|                       The Water Man|        2021|    PG|  92 min|
|                                Kate|        2021|     R| 106 min|
+------------------------------------+------------+------+--------+
only showing top 8 rows


In [0]:
# CELDA 17: Validación 3B — SELECT equivalente SQL
print("🔍 VALIDACIÓN 3B: SELECT con filtros (Spark SQL)")
print("   ↑ Misma consulta que celda anterior — resultados idénticos")
print("-" * 52)
spark.sql(f"""
SELECT title, release_year, rating, duration
FROM {TABLE_NAME}
WHERE type = 'Movie'
  AND release_year >= 2015
  AND country LIKE '%United States%'
ORDER BY release_year DESC
LIMIT 8
""").show(truncate=40)

🔍 VALIDACIÓN 3B: SELECT con filtros (Spark SQL)
   ↑ Misma consulta que celda anterior — resultados idénticos
----------------------------------------------------
+------------------------------------+------------+------+--------+
|                               title|release_year|rating|duration|
+------------------------------------+------------+------+--------+
|                       Final Account|        2021| PG-13|  94 min|
|                       The Water Man|        2021|    PG|  92 min|
|Monster Hunter: Legends of the Guild|        2021| TV-PG|  59 min|
|                                Kate|        2021|     R| 106 min|
|              Untold: Breaking Point|        2021| TV-MA|  80 min|
|                        The Starling|        2021| PG-13| 104 min|
|                          Sweet Girl|        2021|     R| 110 min|
|                 The Kissing Booth 3|        2021| TV-14| 114 min|
+------------------------------------+------------+------+--------+



In [0]:
# CELDA 18: Validación 4A — GROUP BY PySpark
print("🔍 VALIDACIÓN 4A: GROUP BY — Distribución por tipo (PySpark)")
(
    df_netflix
    .groupBy("type")
    .agg(
        F.count("*").alias("total"),
        F.round(F.count("*") / total_rows * 100, 2).alias("porcentaje")
    )
    .orderBy(F.col("total").desc())
).show()

print("\n  Top 10 países con más contenido:")
(
    df_netflix
    .filter(F.col("country").isNotNull())
    .groupBy("country")
    .count()
    .orderBy(F.col("count").desc())
).show(10, truncate=40)

🔍 VALIDACIÓN 4A: GROUP BY — Distribución por tipo (PySpark)
+-------+-----+----------+
|   type|total|porcentaje|
+-------+-----+----------+
|  Movie| 6131|     69.62|
|TV Show| 2676|     30.38|
+-------+-----+----------+


  Top 10 países con más contenido:
+--------------+-----+
|       country|count|
+--------------+-----+
| United States| 2818|
|         India|  972|
|United Kingdom|  419|
|         Japan|  245|
|   South Korea|  199|
|        Canada|  181|
|         Spain|  145|
|        France|  124|
|        Mexico|  110|
|         Egypt|  106|
+--------------+-----+
only showing top 10 rows


In [0]:
# CELDA 19: Validación 4B — GROUP BY SQL
print("🔍 VALIDACIÓN 4B: GROUP BY (Spark SQL)")
spark.sql(f"""
SELECT
    type,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS porcentaje
FROM {TABLE_NAME}
GROUP BY type
ORDER BY total DESC
""").show()

🔍 VALIDACIÓN 4B: GROUP BY (Spark SQL)
+-------+-----+----------+
|   type|total|porcentaje|
+-------+-----+----------+
|  Movie| 6131|     69.62|
|TV Show| 2676|     30.38|
+-------+-----+----------+



In [0]:
# CELDA 20: GROUP BY por año (SQL)
print("🔍 Producción por año (últimos 15 años):")
spark.sql(f"""
SELECT
    release_year,
    COUNT(*) AS total,
    SUM(CASE WHEN type='Movie'   THEN 1 ELSE 0 END) AS peliculas,
    SUM(CASE WHEN type='TV Show' THEN 1 ELSE 0 END) AS series
FROM {TABLE_NAME}
WHERE release_year >= 2008
GROUP BY release_year
ORDER BY release_year DESC
LIMIT 15
""").show()

🔍 Producción por año (últimos 15 años):
+------------+-----+---------+------+
|release_year|total|peliculas|series|
+------------+-----+---------+------+
|        2021|  592|      277|   315|
|        2020|  953|      517|   436|
|        2019| 1030|      633|   397|
|        2018| 1147|      767|   380|
|        2017| 1032|      767|   265|
|        2016|  902|      658|   244|
|        2015|  560|      398|   162|
|        2014|  352|      264|    88|
|        2013|  288|      225|    63|
|        2012|  237|      173|    64|
|        2011|  185|      145|    40|
|        2010|  194|      154|    40|
|        2009|  152|      118|    34|
|        2008|  136|      113|    23|
+------------+-----+---------+------+



In [0]:
# CELDA 21: Validación 5 — Conteos e integridad
print("🔍 VALIDACIÓN 5: Conteos y verificación de PK")
print("=" * 52)
total      = df_netflix.count()
ids_unicos = df_netflix.select("show_id").distinct().count()
peliculas  = df_netflix.filter(F.col("type") == "Movie").count()
series     = df_netflix.filter(F.col("type") == "TV Show").count()

print(f"  📊 Total registros     : {total:,}")
print(f"  🔑 IDs únicos (show_id): {ids_unicos:,}")
print(f"  🎬 Películas           : {peliculas:,}")
print(f"  📺 Series              : {series:,}")

if total == ids_unicos:
    print("\n  ✅ INTEGRIDAD OK: show_id es único (actúa como PK)")
else:
    print(f"\n  ⚠️  {total - ids_unicos} IDs duplicados detectados")
print("=" * 52)

🔍 VALIDACIÓN 5: Conteos y verificación de PK
  📊 Total registros     : 8,807
  🔑 IDs únicos (show_id): 8,807
  🎬 Películas           : 6,131
  📺 Series              : 2,676

  ✅ INTEGRIDAD OK: show_id es único (actúa como PK)


In [0]:
# CELDA 22: Clasificaciones de contenido (SQL)
print("🔍 Distribución por clasificación de audiencia:")
spark.sql(f"""
SELECT
    COALESCE(rating, 'SIN CLASIFICACIÓN') AS clasificacion,
    COUNT(*) AS total,
    ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM {TABLE_NAME}), 2) AS porcentaje
FROM {TABLE_NAME}
GROUP BY rating
ORDER BY total DESC
LIMIT 12
""").show()

🔍 Distribución por clasificación de audiencia:
+-------------+-----+----------+
|clasificacion|total|porcentaje|
+-------------+-----+----------+
|        TV-MA| 3207|     36.41|
|        TV-14| 2160|     24.53|
|        TV-PG|  863|      9.80|
|            R|  799|      9.07|
|        PG-13|  490|      5.56|
|        TV-Y7|  334|      3.79|
|         TV-Y|  307|      3.49|
|           PG|  287|      3.26|
|         TV-G|  220|      2.50|
|           NR|   80|      0.91|
|            G|   41|      0.47|
|     TV-Y7-FV|    6|      0.07|
+-------------+-----+----------+



In [0]:
# CELDA 23: Top directores (PySpark)
print("🎬 Top 15 directores con más títulos en Netflix:")
(
    df_netflix
    .filter(F.col("director").isNotNull())
    .groupBy("director")
    .agg(F.count("*").alias("total_titulos"))
    .orderBy(F.col("total_titulos").desc())
).show(15, truncate=40)

🎬 Top 15 directores con más títulos en Netflix:
+----------------------+-------------+
|              director|total_titulos|
+----------------------+-------------+
|         Rajiv Chilaka|           19|
|Raúl Campos, Jan Suter|           18|
|           Suhas Kadav|           16|
|          Marcus Raboy|           16|
|             Jay Karas|           14|
|   Cathy Garcia-Molina|           13|
|           Jay Chapman|           12|
|       Youssef Chahine|           12|
|       Martin Scorsese|           12|
|      Steven Spielberg|           11|
|      Don Michael Paul|           10|
|          David Dhawan|            9|
|       Fernando Ayllón|            8|
|     Quentin Tarantino|            8|
|        Yılmaz Erdoğan|            8|
+----------------------+-------------+
only showing top 15 rows


## 4.2 Resumen de Validaciones

| # | Validación | Herramienta | Resultado |
|---|---|---|---|
| 1 | Esquema / Metadatos | `printSchema()` | 12 columnas, tipos correctos |
| 2 | Metadatos SQL | `DESCRIBE TABLE` | Coincide con StructType |
| 3 | Estadísticas | `df.describe()` | Rangos y conteos válidos |
| 4 | Nulos | PySpark `isNull()` | Director y country con más nulos |
| 5 | SELECT + filtros | PySpark + SQL | Resultados idénticos ✅ |
| 6 | GROUP BY tipo | PySpark + SQL | ~70% Movies, ~30% TV Shows |
| 7 | GROUP BY año | SQL | Producción creciente desde 2015 |
| 8 | Unicidad PK | `count()` | total == ids_unicos ✅ |
| 9 | Clasificaciones | SQL COALESCE | TV-MA y TV-14 dominan |
| 10 | Top directores | PySpark groupBy | Raúl Campos lidera |

> ✅ Todas las validaciones confirman que los datos fueron cargados correctamente.

---
# 5. Análisis: SQL vs Spark

## 5.1 Tabla Comparativa

| Criterio | SQL (Spark SQL) | Spark (PySpark) |
|---|---|---|
| **Curva de aprendizaje** | ✅ Baja — familiar para quien conoce SQL | ⚠️ Media — requiere conocer la API DataFrame |
| **Expresividad declarativa** | ✅ Alta — SELECT, JOIN, GROUP BY, WINDOW | ✅ Alta — encadenamiento de transformaciones |
| **Integración BI** | ✅ Excelente — Tableau, Power BI via JDBC | ⚠️ Requiere exportación adicional |
| **Pipelines complejos** | ⚠️ Limitado en lógica condicional compleja | ✅ Ideal — integra Python, MLlib, Streaming |
| **Funciones personalizadas** | ⚠️ Requiere registrar UDFs explícitamente | ✅ Natural — funciones Python con `udf()` |
| **Rendimiento** | ✅ Catalyst optimizer automático | ✅ Mismo motor + control manual del plan |
| **Legibilidad** | ✅ Autodocumentado, legible para no técnicos | ⚠️ Más verboso, requiere comentarios |
| **Portabilidad** | ✅ Portable entre Spark, Hive, BigQuery | ⚠️ Específico del ecosistema Spark |
| **Depuración** | ⚠️ Difícil en consultas largas | ✅ Inspección paso a paso del DataFrame |
| **Escalabilidad** | ✅ Distribuido sobre el clúster | ✅ Igual — mismo motor de ejecución |

---

## 5.2 ¿Cuándo usar cada uno?

**Usar SQL cuando:**
- Análisis exploratorio rápido sobre datos ya cargados
- Reportes y dashboards conectados a herramientas BI
- El equipo no tiene formación en programación

**Usar PySpark cuando:**
- Pipelines ETL/ELT con múltiples transformaciones
- Machine Learning con MLlib
- Procesamiento en tiempo real con Structured Streaming
- Lógica de negocio compleja con control de flujo

> 💡 **Buena práctica:** SQL para consultas de negocio, PySpark para ingeniería de datos. Databricks permite combinar ambos en el mismo notebook.

In [0]:
# CELDA 24: Demostración práctica — lógica condicional (PySpark vs SQL)
# Nota: En Windows con Python 3.12, las UDFs tienen limitaciones.
# Usamos funciones nativas de Spark (when/otherwise) que son más
# eficientes y portables que las UDFs de Python.

print("📊 Categoría por duración — usando when/otherwise (PySpark nativo):")
print("   👆 Esta lógica condicional compleja es más natural en PySpark que en SQL")
print("-" * 52)

from pyspark.sql import functions as F

(
    df_netflix
    .withColumn(
        "categoria_duracion",
        F.when(F.col("duration").isNull(), "Desconocida")
        .when(F.col("duration").contains("Season"), "Serie")
        .when(
            F.regexp_extract(F.col("duration"), r"(\d+)", 1).cast("int") < 60,
            "Corta (<1h)"
        )
        .when(
            F.regexp_extract(F.col("duration"), r"(\d+)", 1).cast("int") <= 120,
            "Media (1-2h)"
        )
        .otherwise("Larga (>2h)")
    )
    .groupBy("categoria_duracion")
    .count()
    .orderBy(F.col("count").desc())
    .show()
)

print("\n💡 Ventaja PySpark: esta lógica condicional encadenada")
print("   equivale en SQL a múltiples CASE WHEN anidados,")
print("   pero es más legible y mantenible en código Python.")

📊 Categoría por duración — usando when/otherwise (PySpark nativo):
   👆 Esta lógica condicional compleja es más natural en PySpark que en SQL
----------------------------------------------------
+------------------+-----+
|categoria_duracion|count|
+------------------+-----+
|      Media (1-2h)| 4528|
|             Serie| 2676|
|       Larga (>2h)| 1142|
|       Corta (<1h)|  458|
|       Desconocida|    3|
+------------------+-----+


💡 Ventaja PySpark: esta lógica condicional encadenada
   equivale en SQL a múltiples CASE WHEN anidados,
   pero es más legible y mantenible en código Python.


---
# 6. Conclusiones

## 6.1 Logros del Taller

1. ✅ **Esquema diseñado** con `StructType` y DDL Spark SQL con diccionario de datos y diagrama Mermaid.
2. ✅ **Infraestructura configurada** en Databricks CE — Spark 3.4.1, Python 3.10, DBFS documentado.
3. ✅ **Datos ingeridos** desde Kaggle, almacenados en DBFS y persistidos como tabla Delta Lake.
4. ✅ **10 validaciones** realizadas comparando PySpark y Spark SQL con resultados coincidentes.
5. ✅ **Análisis SQL vs Spark** con tabla comparativa de 10 criterios y demo práctica con UDF.

## 6.2 Aprendizajes Clave

- **Databricks CE** permite aprender Big Data real sin costo, con limitación de Single Node.
- **Delta Lake** agrega capacidades ACID que no están en CSV/Parquet simples.
- **Esquema explícito** es mejor práctica que `inferSchema=True` — más rápido y confiable.
- **SQL y PySpark son complementarios** — la madurez del ingeniero de datos está en saber cuándo usar cada uno.
- El dataset evidenció **problemas reales de calidad** (nulos en director, country) frecuentes en datos reales.

## 6.3 Trabajo Futuro

- Convertir `date_added` a `DateType` con `to_date()`
- Normalizar `country` (múltiples países por fila)
- Explotar `listed_in` con `explode()` para análisis de géneros
- Visualizar tendencias con `matplotlib` o `display()` en Databricks

---

*Notebook desarrollado para la Actividad 2 — Infraestructura y Bases de Datos en la Nube.*  
*Plataforma: Databricks Community Edition | Dataset: Netflix Shows (Kaggle)*